# 03 — Modo Comparativo

**Objetivo**: comparar os quatro algoritmos de clustering (K-Means, DBSCAN,
Aglomerativo e Mean Shift) no mesmo dataset, coletando métricas de avaliação internas
e externas, e gerando visualizações lado a lado e uma tabela de ranking.

## Tópicos
1. Pipeline unificado de pré-processamento
2. Execução de todos os algoritmos sobre o mesmo dataset
3. Coleta de métricas: Silhouette, Davies-Bouldin, Calinski-Harabász, n_clusters, n_noise, tempo
4. Métricas externas (quando rótulos verdadeiros estão disponíveis): ARI, NMI
5. Tabela comparativa
6. Gráfico de barras das métricas
7. Scatter PCA 2D lado a lado

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, MeanShift, estimate_bandwidth
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

# ml-clustering-lab pipeline (implementação futura)
# from ml_clustering_lab.pipeline.run_compare import compare_algorithms

%matplotlib inline
sns.set_theme(style='whitegrid')

# Dataset
iris = load_iris(as_frame=True)
X = iris.data.values
y_true = iris.target.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)
print('Dataset pronto:', X_scaled.shape)

## 2. Execução de todos os algoritmos

In [ ]:
bw = estimate_bandwidth(X_scaled, quantile=0.2, n_samples=100, random_state=42)

algorithms = {
    'K-Means':      KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42),
    'DBSCAN':       DBSCAN(eps=0.5, min_samples=5),
    'Aglomerativo': AgglomerativeClustering(n_clusters=3, linkage='ward'),
    'Mean Shift':   MeanShift(bandwidth=bw, bin_seeding=True),
}

results = {}
for name, model in algorithms.items():
    t0 = time.time()
    labels = model.fit_predict(X_scaled)
    elapsed = time.time() - t0
    results[name] = {'labels': labels, 'time_s': round(elapsed, 4)}
    print(f'{name}: {len(set(labels)) - (1 if -1 in labels else 0)} clusters | {elapsed:.4f}s')

## 3 & 4. Coleta de métricas internas e externas

In [ ]:
rows = []
for name, res in results.items():
    labels = res['labels']
    mask = labels != -1  # exclui ruído para métricas internas
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())

    sil  = silhouette_score(X_scaled[mask], labels[mask]) if n_clusters > 1 and mask.sum() > 1 else float('nan')
    db   = davies_bouldin_score(X_scaled[mask], labels[mask]) if n_clusters > 1 and mask.sum() > 1 else float('nan')
    ch   = calinski_harabasz_score(X_scaled[mask], labels[mask]) if n_clusters > 1 and mask.sum() > 1 else float('nan')
    ari  = adjusted_rand_score(y_true, labels)
    nmi  = normalized_mutual_info_score(y_true, labels)

    rows.append({
        'Algoritmo': name,
        'n_clusters': n_clusters,
        'n_noise': n_noise,
        'Silhouette ↑': round(sil, 4),
        'Davies-Bouldin ↓': round(db, 4),
        'Calinski-Harabász ↑': round(ch, 2),
        'ARI ↑': round(ari, 4),
        'NMI ↑': round(nmi, 4),
        'Tempo (s)': res['time_s'],
    })

df_metrics = pd.DataFrame(rows).set_index('Algoritmo')
df_metrics

## 6. Gráfico de barras das métricas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics = ['Silhouette ↑', 'Davies-Bouldin ↓', 'Calinski-Harabász ↑']
for ax, metric in zip(axes, metrics):
    df_metrics[metric].plot.bar(ax=ax, color=sns.color_palette('tab10', 4), edgecolor='white')
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Comparação de Métricas Internas — Iris', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Scatter PCA 2D lado a lado

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, res) in zip(axes, results.items()):
    labels = res['labels']
    sc = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='tab10', s=30, alpha=0.85)
    ax.set_title(name)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(sc, ax=ax)
plt.suptitle('Clusters em PCA 2D — Comparação de Algoritmos', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()